# CSE6242 - HW3 - Q1

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> remove any comment that says "#export" because that will crash the autograder in Gradescope. We use this comment to export your code in these cells for grading.
</div>

Pyspark Imports

In [1]:
#export
### DO NOT MODIFY THIS CELL ###
import pyspark
from pyspark.sql import SQLContext
from pyspark.sql.functions import hour, when, col, date_format, to_timestamp, ceil, coalesce
from pyspark.sql.functions import *

Initialize PySpark Context

In [2]:
### DO NOT MODIFY THIS CELL ###
sc = pyspark.SparkContext(appName="HW3-Q1")
sqlContext = SQLContext(sc)

C:\Users\103ye\AppData\Roaming\Python\Python311\site-packages\pyspark\sql\context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Define function for loading data

In [3]:
### DO NOT MODIFY THIS CELL ###
def load_data():
    df = sqlContext.read.option("header",True) \
     .csv("yellow_tripdata_2019-01_short.csv")
    return df

### Q1.1

Perform data casting to clean incoming dataset

In [4]:
#export
def clean_data(df):
    '''
    input: df a dataframe
    output: df a dataframe with the all the original columns
    '''
    
    # START YOUR CODE HERE ---------
    
    df = df.withColumn("passenger_count", col("passenger_count").cast("Integer"))
    df = df.withColumn("total_amount", col("total_amount").cast("Float"))
    df = df.withColumn("tip_amount", col("tip_amount").cast("Float"))
    df = df.withColumn("trip_distance", col("trip_distance").cast("Float"))
    df = df.withColumn("fare_amount", col("fare_amount").cast("Float"))
    df = df.withColumn("tpep_pickup_datetime", col("tpep_pickup_datetime").cast("Timestamp"))
    df = df.withColumn("tpep_dropoff_datetime", col("tpep_dropoff_datetime").cast("Timestamp"))
    # df.select(['passenger_count', 'total_amount', 'tip_amount', 'trip_distance', 'fare_amount', 'tpep_pickup_datetime', 'tpep_dropoff_datetime']).printSchema()

    # END YOUR CODE HERE -----------
    
    return df

### Q1.2

Find rate per person for based on how many passengers travel between pickup and dropoff locations. 

In [21]:
#export
def common_pair(df):
    '''
    input: df a dataframe
    output: df a dataframe with following columns:
            - PULocationID
            - DOLocationID
            - total_passenger_count
            - per_person_rate
            
    per_person_rate is the total_amount per person for a given pair.
    
    '''
    
    # START YOUR CODE HERE ---------

    # Remove null or invalid values
    df = df.withColumn("passenger_count", coalesce("passenger_count"))
    df = df.withColumn("total_amount", coalesce("total_amount"))
    
    # Group by the pickup and drop-off locations
    df = df.groupBy("PULocationID", "DOLocationID").agg(
        sum("passenger_count").alias("total_passenger_count"),   # Sum of passenger count
        sum("total_amount").alias("total_amount")                # Sum of total amount
    )
    
    # Calculate per person rate as total_amount / total_passenger_count
    df = df.withColumn("per_person_rate", col("total_amount") / col("total_passenger_count"))
    
    # Filter out rows where the pickup location equals drop-off location
    df = df.filter(col("PULocationID") != col("DOLocationID"))
    
    # Sort by total_passenger_count descending, and then per_person_rate descending
    df = df.orderBy(["total_passenger_count", "per_person_rate"], ascending=[False, False])
    
    # Select top 10 rows
    df = df.limit(10)
    
    # Select only the necessary columns
    df = df.select("PULocationID", "DOLocationID", "total_passenger_count", "per_person_rate")
    
    # END YOUR CODE HERE -----------
    
    return df

### Q1.3

Find trips which trip distances generate the highest tip percentage.

In [22]:
#export
def distance_with_most_tip(df):
    '''
    input: df a dataframe
    output: df a dataframe with following columns:
            - trip_distance
            - tip_percent
            
    trip_percent is the percent of tip out of fare_amount
    
    '''
    
    # START YOUR CODE HERE ---------

    # Filter trips with fare_amount > 2.00 and trip_distance > 0
    df = df.filter((col("fare_amount") > 2.00) & (col("trip_distance") > 0))
    
    # Calculate tip_percent (tip_amount * 100 / fare_amount)
    df = df.withColumn("tip_percent", (col("tip_amount") * 100 / col("fare_amount")))
    
    # Round trip distances up to the nearest mile
    df = df.withColumn("trip_distance", ceil(col("trip_distance")))
    
    # Compute the average tip percent per rounded trip distance
    df = df.groupBy("trip_distance").agg(
        (sum(col("tip_percent")) / coalesce(sum(when(col("tip_percent").isNotNull(), 1)), col("trip_distance"))).alias("tip_percent")
    )
    
    # Sort by tip_percent in descending order and select the top 15
    df = df.orderBy(col("tip_percent").desc()).limit(15)
    
    # END YOUR CODE HERE -----------
    
    return df

### Q1.4

Determine the average speed at different times of day.

In [32]:
#export
def time_with_most_traffic(df):
    '''
    input: df a dataframe
    output: df a dataframe with following columns:
            - time_of_day
            - am_avg_speed
            - pm_avg_speed
            
    am_avg_speed and pm_avg_speed are the average trip distance / average trip time calculated for each hour
    
    '''
    
    # START YOUR CODE HERE ---------

    # Extract the hour from the pickup datetime
    df = df.withColumn("hour", hour(col("tpep_pickup_datetime")))
    
    # Calculate trip duration in seconds
    # df = df.withColumn(
    #     "trip_duration",
    #     (to_timestamp(col("tpep_dropoff_datetime")).cast("long") - to_timestamp(col("tpep_pickup_datetime")).cast("long")
    # ))
    # Calculate trip duration in seconds
    df = df.withColumn(
        "trip_duration",
        (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))
    ))

    # print(df.head())
    
    # Filter out rows with zero or null trip durations
    df = df.filter((col("trip_duration") > 0) & (col("trip_duration").isNotNull()))
    
    # Calculate average trip distance and average trip duration for each hour
    df_grouped = df.groupBy("hour").agg(
        avg(col("trip_distance")).alias("avg_distance"),
        avg(col("trip_duration")).alias("avg_duration")
    )
    
    # Calculate average speed as average distance / average duration (distance per hour)
    df_grouped = df_grouped.withColumn(
        "avg_speed",
        (col("avg_distance") / col("avg_duration")) * 3600  # Convert to distance per hour
    )
    
    # Define AM and PM periods
    df_grouped = df_grouped.withColumn(
        "period",
        when(col("hour") < 12, "AM").otherwise("PM")
    )
    
    # Convert hour to 12-hour format
    df_grouped = df_grouped.withColumn(
        "time_of_day",
        when(col("hour") == 0, lit(0)).otherwise(col("hour") % 12 )
    )
    
    # Pivot data to separate AM and PM speeds
    df_pivot = df_grouped.groupBy("time_of_day").pivot("period", ["AM", "PM"]).agg(
        avg(col("avg_speed"))
    )
    
    # Rename columns
    df_pivot = df_pivot.withColumnRenamed("AM", "am_avg_speed").withColumnRenamed("PM", "pm_avg_speed")
    
    # Sort by time_of_day
    df = df_pivot.orderBy("time_of_day")
    
    # END YOUR CODE HERE -----------
    
    return df

## The below cells are for you to investigate your solutions and will not be graded

In [24]:
df = load_data()
df = clean_data(df)

In [25]:
common_pair(df).show()

+------------+------------+---------------------+------------------+
|PULocationID|DOLocationID|total_passenger_count|   per_person_rate|
+------------+------------+---------------------+------------------+
|         239|         238|                   62|  4.26274198870505|
|         237|         236|                   60| 4.482500068346659|
|         263|         141|                   52|3.4190384974846473|
|         161|         236|                   42| 5.368571440378825|
|         148|          79|                   42| 4.711904752822149|
|         142|         238|                   39|  5.05487182812813|
|         141|         236|                   37| 4.355675723101641|
|         239|         143|                   37| 4.252162224537617|
|         239|         142|                   35| 3.817714350564139|
|          79|         170|                   34| 6.394705884596881|
+------------+------------+---------------------+------------------+



In [26]:
distance_with_most_tip(df).show()

+-------------+------------------+
|trip_distance|       tip_percent|
+-------------+------------------+
|            1|17.129815971513313|
|            2|15.815527155632552|
|           17|15.796441782308916|
|           20| 15.11240992123345|
|            3|14.886705727113446|
|            6|14.579695131601051|
|            5|14.245405861990653|
|            4|13.831569507473274|
|            9|13.814476557648435|
|            8|12.072596772433315|
|           19|11.952632334985276|
|           10|11.880490518902954|
|            7| 10.80057562837643|
|           21|10.739019886973427|
|           18|10.696822158448429|
+-------------+------------------+



In [33]:
time_with_most_traffic(df).show()

+-----------+------------------+------------------+
|time_of_day|      am_avg_speed|      pm_avg_speed|
+-----------+------------------+------------------+
|          0| 9.377696196631257|              NULL|
|          1|10.845483413697357| 5.125214305177561|
|          3|              NULL|               0.0|
|          4|              NULL|               0.0|
|          5|              NULL|0.5137660239764732|
|          6|              NULL| 9.989847870647605|
|          7|              NULL|0.1841530549041771|
|          8|              NULL|0.5183127622697897|
|         10|              NULL|0.6147483972627695|
|         11|              NULL|4.6509582852075795|
+-----------+------------------+------------------+

